In [ ]:
import os
import pandas as pd

# Preprocessing

In this notebook, we:
- check for exactly duplicated rows and delete them
- remove residue pairs that have only "Unclassified" interaction type
- encode interaction types as a binary vector
- save to file the processed dataset as  "classification_ring/data/processed/data_ml.parquet".

#### 1. Load and inspect the dataset

In [3]:
# Combine all PDBs into a single dataframe
dfs = []
for filename in os.listdir('classification_ring/data/features_ring'):
    dfs.append(pd.read_csv('classification_ring/data/features_ring/' + filename, sep='\t'))
df = pd.concat(dfs)
print(df.head())
print(df.shape) # Check the shape of the dataframe (number of rows and columns)

# Check the unique values in the 'Interaction' column to understand the distribution of interaction types
print("\nUnique interaction labels:")
print(df["Interaction"].value_counts(dropna=False))

  pdb_id s_ch  s_resi s_ins s_resn s_ss8  s_rsa  s_phi  s_psi   s_a1  ...  \
0   1u9c    A      71            A     H  0.434 -1.145 -0.683 -0.591  ...   
1   1u9c    A     172            S     H  0.354 -1.001 -0.749 -0.228  ...   
2   1u9c    A     172            S     H  0.354 -1.001 -0.749 -0.228  ...   
3   1u9c    A     155            E     H  0.351 -1.048 -0.782  1.357  ...   
4   1u9c    A     184            G     -  0.250 -2.102 -3.018 -0.384  ...   

   t_phi  t_psi   t_a1   t_a2   t_a3   t_a4   t_a5  t_3di_state t_3di_letter  \
0 -2.580  1.037  0.336 -0.417 -1.673 -1.474 -0.078          5.0            F   
1 -1.036 -0.619 -1.019 -0.987 -1.505  1.266 -0.912         17.0            R   
2 -1.036 -0.619 -1.019 -0.987 -1.505  1.266 -0.912         17.0            R   
3 -1.153 -0.666  1.538 -0.055  1.502  0.440  2.897         17.0            R   
4 -1.486 -0.626  0.931 -0.179 -3.005 -0.503 -1.853         19.0            T   

  Interaction  
0         NaN  
1       HBOND  
2       

A residue is uniquely identified by the combination of 

- pdb_id: PDB entry
- s/t_ch: chain index
- s/t_resi: residue index in the chain
- s/t_ins: insertion code (usually ' ')
- s/t_resn: residue name (one letter code)


#### 2. Check data integrity: exactly duplicated rows and/or columns with missing values

In [8]:
# Check for missing values in each column of the dataframe (not only the 'Interaction' column) to understand the completeness of the data
print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False).head(20)) # .isna().sum() counts the number of missing values in each column, and .sort_values(ascending=False) sorts the columns by the number of missing values in descending order. The .head(20) method displays the top 20 columns with the most missing values.

# Check for any exactly duplicated rows and remove them.
print("\nDuplicated rows:")
print(df.duplicated().sum())


Missing values:
Interaction     1089547
t_3di_letter      44036
t_3di_state       44036
s_3di_letter      37025
s_3di_state       37025
t_psi             21474
s_phi             17807
s_psi              6736
t_phi              6167
t_rsa                75
s_rsa                63
t_ss8                56
s_ss8                31
t_a1                  0
t_a2                  0
t_a3                  0
t_resn                0
t_a5                  0
t_a4                  0
pdb_id                0
dtype: int64

Duplicated rows:
0


In [11]:
# check for duplicate rows based on the columns that define the interaction pair AND interaction type.

mask_cols = [
    'pdb_id',
    's_ch', 's_resi', 's_ins', 's_resn',
    't_ch', 't_resi', 't_ins', 't_resn', 'Interaction'
]

mask = df.duplicated(subset=mask_cols, keep=False)
print("\nDuplicated rows based on interaction pair and type:")
print(mask.sum())


Duplicated rows based on interaction pair and type:
0


In [16]:
pair_cols =  [
    'pdb_id',
    's_ch', 's_resi', 's_ins', 's_resn',
    't_ch', 't_resi', 't_ins', 't_resn'
]


dup_counts = df.groupby(pair_cols).size().sort_values(ascending = False) # Count occurences of each unique residue pair


dup_pairs = dup_counts[dup_counts > 1]

print(f"Number of duplicated residue pairs: {len(dup_pairs)}")

# Manually inspect one of the duplicated pairs to understand why it appears multiple times (spoiler: it is because it has multiple interaction types, which is expected and correct).
example_pair = dup_pairs.index[0]

mask = (
    (df['pdb_id'] == example_pair[0]) &
    (df['s_ch']   == example_pair[1]) &
    (df['s_resi'] == example_pair[2]) &
    (df['s_ins']  == example_pair[3]) &
    (df['s_resn'] == example_pair[4]) &
    (df['t_ch']   == example_pair[5]) &
    (df['t_resi'] == example_pair[6]) &
    (df['t_ins']  == example_pair[7]) &
    (df['t_resn'] == example_pair[8])
)

print("Example of a residue pair appearing multiple times: ")
df.loc[mask]

Number of duplicated residue pairs: 401711
Example of a residue pair appearing multiple times: 


,pdb_id,s_ch,s_resi,s_ins,s_resn,s_ss8,s_rsa,s_phi,s_psi,s_a1,...,t_phi,t_psi,t_a1,t_a2,t_a3,t_a4,t_a5,t_3di_state,t_3di_letter,Interaction
146,2r6v,A,5,,R,G,0.165,-1.144,-0.313,1.538,...,-1.498,2.773,0.26,0.83,3.097,-0.838,1.512,NaN,NaN,PICATION
147,2r6v,A,5,,R,G,0.165,-1.144,-0.313,1.538,...,-1.498,2.773,0.26,0.83,3.097,-0.838,1.512,NaN,NaN,HBOND
148,2r6v,A,5,,R,G,0.165,-1.144,-0.313,1.538,...,-1.498,2.773,0.26,0.83,3.097,-0.838,1.512,NaN,NaN,PIHBOND
149,2r6v,A,5,,R,G,0.165,-1.144,-0.313,1.538,...,-1.498,2.773,0.26,0.83,3.097,-0.838,1.512,NaN,NaN,VDW


----

#### 2. Cleaning


1. Convert missing Interaction values to "Unclassified"
2. Remove purely Unclassified residue pairs, but ignore Unclassified when a pair also has valid labels
4. Build one row per residue pair with multi-label targets
5. Only then inspect/drop/impute missing feature values

In [17]:
df["Interaction"] = df["Interaction"].fillna("Unclassified")
print(df["Interaction"].value_counts())

Interaction
Unclassified    1089547
HBOND           1055929
VDW              737061
PIPISTACK         38283
IONIC             35391
PICATION           8885
SSBOND             2100
PIHBOND            1790
Name: count, dtype: int64


In [18]:
# -----------------------------
# 2. Define residue-pair identity columns
# -----------------------------

pair_cols = [
    "pdb_id",
    "s_ch", "s_resi", "s_ins", "s_resn",
    "t_ch", "t_resi", "t_ins", "t_resn"
]

# -----------------------------
# 3. Remove purely Unclassified residue pairs
#    but keep pairs that have at least one valid label
# -----------------------------

# Get the list of valid labels (excluding "Unclassified") 
valid_labels = sorted([
    label for label in df["Interaction"].unique()
    if label != "Unclassified"
])

print("Valid labels:", valid_labels)

# .transform() is used to broadcast the result of the groupby operation back to the original dataframe,
#  so that we can filter the rows based on whether the pair has at least one valid label.
pair_has_valid_label = (
    df.groupby(pair_cols)["Interaction"]
      .transform(lambda labels: (labels != "Unclassified").any())
)

df_clean = df[pair_has_valid_label].copy()

# Now remove Unclassified rows from pairs that also have valid labels
df_clean = df_clean[df_clean["Interaction"] != "Unclassified"].copy()

print("Original rows:", len(df))
print("Clean rows:", len(df_clean))
print(df_clean["Interaction"].value_counts())

Valid labels: ['HBOND', 'IONIC', 'PICATION', 'PIHBOND', 'PIPISTACK', 'SSBOND', 'VDW']
Original rows: 2968986
Clean rows: 1879439
Interaction
HBOND        1055929
VDW           737061
PIPISTACK      38283
IONIC          35391
PICATION        8885
SSBOND          2100
PIHBOND         1790
Name: count, dtype: int64


In [ ]:
# ---------------------
# Reset the index of the cleaned dataframe to ensure it is sequential and starts from 0,
# drop=True means that the old index is not added as a column in the new dataframe.
# ---------------------
df_clean = df_clean.reset_index(drop=True)
df_clean.head()

,pdb_id,s_ch,s_resi,s_ins,s_resn,s_ss8,s_rsa,s_phi,s_psi,s_a1,...,t_phi,t_psi,t_a1,t_a2,t_a3,t_a4,t_a5,t_3di_state,t_3di_letter,Interaction
0,1u9c,A,172,,S,H,0.354,-1.001,-0.749,-0.228,...,-1.036,-0.619,-1.019,-0.987,-1.505,1.266,-0.912,17.0,R,HBOND
1,1u9c,A,172,,S,H,0.354,-1.001,-0.749,-0.228,...,-1.036,-0.619,-1.019,-0.987,-1.505,1.266,-0.912,17.0,R,VDW
2,1u9c,A,184,,G,-,0.250,-2.102,-3.018,-0.384,...,-1.486,-0.626,0.931,-0.179,-3.005,-0.503,-1.853,19.0,T,HBOND
3,1u9c,A,10,,T,-,0.000,-1.252,2.751,-0.032,...,-2.136,-0.075,1.050,0.302,-3.656,-0.259,-3.242,12.0,M,HBOND
4,1u9c,A,133,,V,T,0.021,-1.076,-0.664,-1.337,...,-1.257,-0.710,-0.032,0.326,2.213,0.908,1.313,15.0,P,VDW


In [20]:
print("\nMissing values:")
print(df_clean.isna().sum().sort_values(ascending=False).head(20))


Missing values:
t_3di_letter    28744
t_3di_state     28744
s_3di_letter    24020
s_3di_state     24020
t_psi           14104
s_phi           10951
s_psi            4484
t_phi            3216
t_rsa               9
pdb_id              0
t_a1                0
t_a3                0
t_a2                0
t_resn              0
t_a4                0
t_a5                0
Interaction         0
t_ss8               0
t_ch                0
t_ins               0
dtype: int64


**IMPORTANT COMMENT**: there still are rows containing missing values. We will use Inputation.

In [21]:
# -----------------------------
# 4. Build multi-label targets
#    one row = one residue pair
# -----------------------------
# Add a dummy value to indicate that this interaction is present
df_clean["label_present"] = 1
df_clean["pdb_id"] = df_clean["pdb_id"].astype(str)

# Build binary label matrix
Y = (
    df_clean
    .pivot_table(
        index=pair_cols,
        columns="Interaction",
        values="label_present",
        aggfunc="max",
        fill_value=0
    ).reset_index()
)
# Remove the column index name created by pivot_table
Y.columns.name = None
print("Feature matrix shape:", Y.shape)
Y.head()

Feature matrix shape: (1452411, 16)


,pdb_id,s_ch,s_resi,s_ins,s_resn,t_ch,t_resi,t_ins,t_resn,HBOND,IONIC,PICATION,PIHBOND,PIPISTACK,SSBOND,VDW
0,1aba,A,2,,F,A,30,,P,1,0,0,0,0,0,1
1,1aba,A,2,,F,A,32,,E,1,0,0,0,0,0,0
2,1aba,A,3,,K,A,69,,F,1,0,0,0,0,0,1
3,1aba,A,4,,V,A,32,,E,1,0,0,0,0,0,0
4,1aba,A,4,,V,A,33,,F,0,0,0,0,0,0,1


In [22]:
# Merge features with multilabel targets
feature_cols = [col for col in df_clean.columns 
                if col not in ["Interaction", "label_present"]]

X = (
    df_clean[feature_cols]
    .drop_duplicates(subset=pair_cols)
    .reset_index(drop=True)
)

data_ml = X.merge(Y, on=pair_cols, how="inner")

label_cols = [col for col in Y.columns if col not in pair_cols]

print("Multi-label dataset shape:", data_ml.shape)
print("Feature matrix shape:", X.shape)
print("Label columns:", label_cols)

data_ml.head()

Multi-label dataset shape: (1452411, 38)
Feature matrix shape: (1452411, 31)
Label columns: ['HBOND', 'IONIC', 'PICATION', 'PIHBOND', 'PIPISTACK', 'SSBOND', 'VDW']


,pdb_id,s_ch,s_resi,s_ins,s_resn,s_ss8,s_rsa,s_phi,s_psi,s_a1,...,t_a5,t_3di_state,t_3di_letter,HBOND,IONIC,PICATION,PIHBOND,PIPISTACK,SSBOND,VDW
0,1u9c,A,172,,S,H,0.354,-1.001,-0.749,-0.228,...,-0.912,17.0,R,1,0,0,0,0,0,1
1,1u9c,A,184,,G,-,0.250,-2.102,-3.018,-0.384,...,-1.853,19.0,T,1,0,0,0,0,0,0
2,1u9c,A,10,,T,-,0.000,-1.252,2.751,-0.032,...,-3.242,12.0,M,1,0,0,0,0,0,0
3,1u9c,A,133,,V,T,0.021,-1.076,-0.664,-1.337,...,1.313,15.0,P,0,0,0,0,0,0,1
4,1u9c,A,146,,G,T,0.548,1.315,0.127,-0.384,...,2.064,6.0,G,1,0,0,0,0,0,0


In [23]:
# Create directory if it does not exist
os.makedirs("classification_ring/data/processed", exist_ok=True)

# Save processed multilabel dataframe
data_ml.to_parquet(
    "classification_ring/data/processed/data_ml.parquet",
    index=False
)
print("Saved successfully.")

Saved successfully.
